In [ ]:
%pip install ucimlrepo

In [49]:
import numpy as np
import pandas as pd

In [ ]:
from ucimlrepo import fetch_ucirepo

chronic_kidney_disease = fetch_ucirepo(id=336)

# data (as pandas dataframes) 
X = chronic_kidney_disease.data.features.copy() 
y = chronic_kidney_disease.data.targets.copy() 
  
# metadata 
print(chronic_kidney_disease.metadata) 
  
# variable information 
print(chronic_kidney_disease.variables) 

In [ ]:
X.head()

In [ ]:
print("There are", X.shape[0], "rows and", X.shape[1], "columns in the dataset.")

In [ ]:
X.info()

In [ ]:
for col in X.select_dtypes(include='object').columns:
    print(col, X[col].unique())


In [ ]:
X['dm'].value_counts(dropna=False)

Object column check: All 10 object columns are categorical: each holds two categories (e.g. yes/no, normal/abnormal) plus missing values (nan). Only dm has a formatting error, '\tno' (“no” with a stray tab), which I’ll fix with .str.strip(). Next, I’ll convert each column to numbers (e.g. yes → 1, no → 0) and handle the missing values after the train/test split.

In [ ]:
X['dm'] = X['dm'].str.strip()
X['dm'].value_counts(dropna=False)

In [58]:
y['class'] = y['class'].str.strip()
y.value_counts(dropna=False)

class 
ckd       250
notckd    150
Name: count, dtype: int64

Encoding categorical columns: converting each category to 0/1 (yes → 1, no → 0; abnormal → 1, normal → 0) before the train/test split. Missing values stay as NaN.

In [59]:
mapping = {
    "yes": 1, "no": 0,
    "abnormal": 1, "normal": 0,
    "present": 1, "notpresent": 0,
    "poor": 1, "good": 0,
}

categorical_cols = X.select_dtypes(include="object").columns
X[categorical_cols] = X[categorical_cols].apply(
    lambda col: col.str.strip().map(mapping)
)

for col in categorical_cols:
    print(f"\n{col}")
    print(X[col].value_counts(dropna=False))


rbc
rbc
0.0    201
NaN    152
1.0     47
Name: count, dtype: int64

pc
pc
0.0    259
1.0     76
NaN     65
Name: count, dtype: int64

pcc
pcc
0.0    354
1.0     42
NaN      4
Name: count, dtype: int64

ba
ba
0.0    374
1.0     22
NaN      4
Name: count, dtype: int64

htn
htn
0.0    251
1.0    147
NaN      2
Name: count, dtype: int64

dm
dm
0.0    261
1.0    137
NaN      2
Name: count, dtype: int64

cad
cad
0.0    364
1.0     34
NaN      2
Name: count, dtype: int64

appet
appet
0.0    317
1.0     82
NaN      1
Name: count, dtype: int64

pe
pe
0.0    323
1.0     76
NaN      1
Name: count, dtype: int64

ane
ane
0.0    339
1.0     60
NaN      1
Name: count, dtype: int64


In [60]:
y = y['class'].map({'ckd': 1, 'notckd': 0})
y.value_counts(dropna=False)

class
1    250
0    150
Name: count, dtype: int64